# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatima-zehra5/ML-internhip/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This notebook builds a transparent, rule-based baseline for prioritising content pages for review. The baseline is designed to be readable, deterministic, and useful as a benchmark before model training.
      

## 1. My rule and its reason codes

**Rule:** Prioritise pages that have meaningful search visibility, are getting older or have not been updated recently, have a useful ranking position to protect, or have relatively thin content. The score is a transparent weighted combination of these signals rather than a fitted model.

**Reason codes:** `stale_visible_page`, `declining_with_demand`, `thin_visible_page`, `page_one_decay_risk`, `low_ctr_visible_page`, `low_engagement_visible_page`, and `general_refresh_review`.

The target label is used only for evaluation. It is not used to construct the score.

In [23]:
import numpy as np
import pandas as pd
from pathlib import Path

RAW_URL = 'https://raw.githubusercontent.com/fatima-zehra5/ML-internhip/main/data/raw/content_refresh_anonymized.csv'

df = pd.read_csv(RAW_URL)

print(f'Loaded rows: {len(df):,}')
print(f'Loaded columns: {len(df.columns)}')
print('Required columns present:', all(c in df.columns for c in [
    'content_id', 'client_id', 'impressions_90d', 'days_since_last_update',
    'avg_position', 'word_count', 'trend_direction', 'ctr', 'sessions_90d',
    'engagement_rate', 'scroll_rate', 'content_age_days'
]))

Loaded rows: 30,000
Loaded columns: 44
Required columns present: True


## 2. Build the ranked queue (writes the CSV)

The score uses four transparent components:

- 40% visibility
- 30% freshness risk
- 25% position opportunity
- 5% content-depth gap

No fitted model and no label-derived feature is used.

In [24]:
def percentile_rank(s):
    s = pd.to_numeric(s, errors='coerce').fillna(0)
    return s.rank(method='average', pct=True).fillna(0)


def normalize(s):
    s = pd.to_numeric(s, errors='coerce').fillna(0)
    lo = s.min()
    hi = s.max()
    if hi == lo:
        return pd.Series(0.0, index=s.index)
    return (s - lo) / (hi - lo)


work = df.copy()

numeric_cols = [
    'impressions_90d', 'days_since_last_update', 'avg_position',
    'word_count', 'content_age_days', 'ctr', 'sessions_90d',
    'engagement_rate', 'scroll_rate'
]

for col in numeric_cols:
    work[col] = pd.to_numeric(work[col], errors='coerce').fillna(0)

work = work.drop_duplicates(subset=['content_id']).reset_index(drop=True)

work['visibility_score'] = percentile_rank(np.log1p(work['impressions_90d']))
work['freshness_risk_score'] = percentile_rank(work['days_since_last_update'])

work['position_opportunity_score'] = (
    (1 - normalize(work['avg_position'].clip(lower=1, upper=50)))
    * work['visibility_score']
    * (work['avg_position'] > 0).astype(int)
)

work['depth_gap_score'] = (
    1 - percentile_rank(work['word_count'])
) * work['visibility_score']

work['baseline_refresh_score'] = (
    0.40 * work['visibility_score']
    + 0.30 * work['freshness_risk_score']
    + 0.25 * work['position_opportunity_score']
    + 0.05 * work['depth_gap_score']
).clip(0, 1)


def reason_codes(row):
    reasons = []

    if row['days_since_last_update'] >= 180 and row['impressions_90d'] >= 500:
        reasons.append('stale_visible_page')

    if str(row['trend_direction']).lower() == 'down' and row['impressions_90d'] >= 100:
        reasons.append('declining_with_demand')

    if 0 < row['word_count'] < 1200 and row['impressions_90d'] >= 250:
        reasons.append('thin_visible_page')

    if 0 < row['avg_position'] <= 10 and row['content_age_days'] >= 180:
        reasons.append('page_one_decay_risk')

    if row['impressions_90d'] >= 500 and 0 < row['avg_position'] <= 20 and row['ctr'] < 0.5:
        reasons.append('low_ctr_visible_page')

    if row['sessions_90d'] >= 30 and (
        (0 < row['engagement_rate'] < 30)
        or (0 < row['scroll_rate'] < 30)
    ):
        reasons.append('low_engagement_visible_page')

    if not reasons:
        reasons.append('general_refresh_review')

    return '|'.join(reasons)


def suggested_action(reason_string):
    reasons = set(reason_string.split('|'))

    if 'thin_visible_page' in reasons:
        return 'expand_and_refresh'
    if 'low_ctr_visible_page' in reasons:
        return 'refresh_and_review_ctr'
    if 'stale_visible_page' in reasons or 'declining_with_demand' in reasons:
        return 'refresh'
    return 'monitor'


work['reason_codes'] = work.apply(reason_codes, axis=1)
work['suggested_action_baseline'] = work['reason_codes'].apply(suggested_action)
work['baseline_rank'] = work['baseline_refresh_score'].rank(
    method='first', ascending=False
).astype(int)

work['is_declining_label'] = (
    work['trend_direction'].astype(str).str.lower().eq('down').astype(int)
)

queue_columns = [
    'content_id', 'client_id', 'baseline_rank', 'baseline_refresh_score',
    'visibility_score', 'freshness_risk_score', 'position_opportunity_score',
    'depth_gap_score', 'reason_codes', 'suggested_action_baseline',
    'is_declining_label', 'impressions_90d', 'clicks_90d', 'sessions_90d',
    'avg_position', 'ctr', 'engagement_rate', 'scroll_rate',
    'content_age_days', 'days_since_last_update', 'word_count',
    'trend_direction'
]

queue = work[queue_columns].sort_values('baseline_rank').reset_index(drop=True)

output_dir = Path('work/outputs')
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / 'baseline_action_score.csv'
queue.to_csv(output_path, index=False)

print(f'Ranked queue rows: {len(queue):,}')
print(f'CSV written to: {output_path}')
print(f'Top score: {queue.baseline_refresh_score.max():.4f}')
print(f'Median score: {queue.baseline_refresh_score.median():.4f}')

Ranked queue rows: 30,000
CSV written to: work/outputs/baseline_action_score.csv
Top score: 0.9412
Median score: 0.4429


## 3. Top-20 review

The top 20 are reviewed as decision-support items. Each row contains an action, reason code, confidence note, and a condition that could make the recommendation wrong.

In [25]:
top20 = queue.head(20).copy()

def confidence_note(row):
    if row['impressions_90d'] >= 3000 and row['baseline_refresh_score'] >= 0.75:
        return 'higher confidence: strong visibility and high baseline score'
    if row['impressions_90d'] >= 500:
        return 'moderate confidence: enough visible demand for review prioritisation'
    return 'lower confidence: limited observed search volume'


def what_would_make_it_wrong(row):
    if row['impressions_90d'] < 100:
        return 'low search volume could make the observed signals unstable'
    if row['avg_position'] == 0:
        return 'missing position data could weaken the position-based interpretation'
    if row['days_since_last_update'] < 90:
        return 'the page may already be reasonably fresh despite other signals'
    return 'the score is directional and may prioritise a page whose underlying issue is not actionable'


top20['confidence_note'] = top20.apply(confidence_note, axis=1)
top20['what_would_make_it_wrong'] = top20.apply(what_would_make_it_wrong, axis=1)

review_columns = [
    'baseline_rank', 'content_id', 'suggested_action_baseline',
    'reason_codes', 'baseline_refresh_score', 'confidence_note',
    'what_would_make_it_wrong', 'impressions_90d', 'avg_position',
    'days_since_last_update'
]

display(top20[review_columns])

print('\nTop-20 action counts:')
print(top20['suggested_action_baseline'].value_counts())

,baseline_rank,content_id,suggested_action_baseline,reason_codes,baseline_refresh_score,confidence_note,what_would_make_it_wrong,impressions_90d,avg_position,days_since_last_update
0,1,content_9532f197bbc8,refresh,declining_with_demand|page_one_decay_risk|low_...,0.941189,higher confidence: strong visibility and high ...,the score is directional and may prioritise a ...,309192,2.0,104
1,2,content_4d1fe5b32dc2,monitor,page_one_decay_risk|low_engagement_visible_page,0.934889,higher confidence: strong visibility and high ...,the score is directional and may prioritise a ...,97999,2.5,104
2,3,content_07f2e7a6f38a,monitor,page_one_decay_risk|low_engagement_visible_page,0.934080,higher confidence: strong visibility and high ...,the score is directional and may prioritise a ...,101078,2.7,104
3,4,content_e5ae436f9a16,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_e...,0.933606,higher confidence: strong visibility and high ...,the score is directional and may prioritise a ...,117741,3.0,104
4,5,content_3430a8b94511,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_e...,0.933559,higher confidence: strong visibility and high ...,the score is directional and may prioritise a ...,152617,3.3,104
5,6,content_cbd93118300b,refresh_and_review_ctr,declining_with_demand|page_one_decay_risk|low_...,0.933263,higher confidence: strong visibility and high ...,the score is directional and may prioritise a ...,145292,3.3,104
6,7,content_9c195417f6ef,monitor,page_one_decay_risk|low_engagement_visible_page,0.932991,higher confidence: strong visibility and high ...,the score is directional and may prioritise a ...,79146,2.5,104
7,8,content_ba2acb4ebd04,monitor,page_one_decay_risk|low_engagement_visible_page,0.931623,higher confidence: strong visibility and high ...,the score is directional and may prioritise a ...,142072,3.6,104
8,9,content_79b25654070a,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_e...,0.931363,higher confidence: strong visibility and high ...,the score is directional and may prioritise a ...,148737,3.7,104
9,10,content_adddad39251c,monitor,page_one_decay_risk|low_engagement_visible_page,0.931124,higher confidence: strong visibility and high ...,the score is directional and may prioritise a ...,129239,3.6,104



Top-20 action counts:
suggested_action_baseline
refresh_and_review_ctr    9
monitor                   8
refresh                   3
Name: count, dtype: int64


## 4. Weak picks + leakage check

The baseline is intentionally directional rather than a claim of causal impact. Weak picks are expected when volume is low or the available measurements are incomplete.

For leakage, the score is constructed only from visibility, freshness, position, and content depth. The label source `trend_direction` is used only after scoring to evaluate the queue. No `trend_pct`, `trend_direction`, or `is_declining_label` value enters the score formula.

In [26]:
score_features = {
    'visibility_score',
    'freshness_risk_score',
    'position_opportunity_score',
    'depth_gap_score'
}

forbidden_features = {
    'trend_pct',
    'trend_direction',
    'is_declining_label'
}

assert score_features.isdisjoint(forbidden_features)

weak_picks = queue[
    (queue['impressions_90d'] < 100)
    | (queue['avg_position'] == 0)
].head(5)

print('LEAKAGE CHECK: PASS')
print('Score components:', ', '.join(sorted(score_features)))
print('Forbidden label-derived fields excluded from score:', ', '.join(sorted(forbidden_features)))

print('\nExample weak picks to inspect:')
display(weak_picks[
    ['baseline_rank', 'content_id', 'baseline_refresh_score',
     'impressions_90d', 'avg_position', 'reason_codes']
])

LEAKAGE CHECK: PASS
Score components: depth_gap_score, freshness_risk_score, position_opportunity_score, visibility_score
Forbidden label-derived fields excluded from score: is_declining_label, trend_direction, trend_pct

Example weak picks to inspect:


,baseline_rank,content_id,baseline_refresh_score,impressions_90d,avg_position,reason_codes
13867,13868,content_07ce98c6085a,0.467560,85,5.3,page_one_decay_risk
14315,14316,content_30eb41dff556,0.457596,84,6.2,page_one_decay_risk
14477,14478,content_460b11dcac6a,0.454192,81,8.0,page_one_decay_risk
14641,14642,content_2e2a634851a0,0.450560,81,10.6,general_refresh_review
14731,14732,content_2bec01911e25,0.448357,57,6.1,page_one_decay_risk


## Evaluation — Precision@K and base rate

Precision@K answers: among the highest-ranked pages, how many are actually labelled as declining? The base rate is printed alongside it so the result has context.

In [27]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()


base_rate = queue['is_declining_label'].mean()

print(f'Base rate of declining label: {base_rate:.3f}')

for k in [20, 50, 100]:
    if len(queue) >= k:
        p_at_k = precision_at_k(
            queue['baseline_refresh_score'],
            queue['is_declining_label'],
            k
        )
        print(f'Precision@{k}: {p_at_k:.3f}')

print('\nInterpretation: the baseline is a transparent prioritisation rule, not a causal model.')

Base rate of declining label: 0.542
Precision@20: 0.350
Precision@50: 0.340
Precision@100: 0.380

Interpretation: the baseline is a transparent prioritisation rule, not a causal model.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries are used as analytical features
- [x] Claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — after saving the completed notebook